# Summary
## Done
- Loaded 48 csv_files from ../data/tfl-cycling-journey-data-2024-2025/raw-2024-2025-csv
- Reading csv's and checking for inconsistent schema (i.e. heading/feature names switching places between csv files)
- Found 11 csv files that have misaligned headers/features
- Checked all heading/feature names (Number, Start date, etc.) across all 48 files are spelled exactly the same - i.e. no mis-spellings like "Nmber".

## Todo:
- Since the heading/feature names have switched places across csv files, I must make sure the data under these headings are in the correct location.
- Need to check if the columns are of unequal length for each csv file (are there any blank/empty cells?) **non-rectangular data?**
- pandas .info() and .describe() for each csv to verify data types, detect missing values, look for "odd" data.

In [1]:
import pandas as pd
import os
import psutil
from pathlib import Path

In [3]:
data_dir = Path("../data/tfl-cycling-journey-data-2024-2025/raw-2024-2025-csv")
csv_files = sorted(data_dir.glob("*.csv"))
print(f"Found {len(csv_files)} files in {data_dir}")

Found 48 files in ../data/tfl-cycling-journey-data-2024-2025/raw-2024-2025-csv


## Checking header and data consistency across thes 48 csv files

In [37]:
#Checking all headers
print(f"Listing ALL {len(csv_files)} csv files")
print()
for f in csv_files:
    #Only loading first few rows to reduce load on cpu
    df = pd.read_csv(f, nrows=0) 
    print(f"File: {f.name}")
    print(list(df.columns))
    print("-" * 160)
pd.set_option('display.max_columns', None) 

Listing ALL 48 csv files

File: 387JourneyDataExtract01Jan2024-14Jan2024.csv
['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']
----------------------------------------------------------------------------------------------------------------------------------------------------------------
File: 388JourneyDataExtract15Jan2024-31Jan2024.csv
['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']
----------------------------------------------------------------------------------------------------------------------------------------------------------------
File: 389JourneyDataExtract01Feb2024-14Feb2024.csv
['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number',

Some **column headings are misaligned** or in the wrong place. I'll have to check each file to make sure the data isn't in the wrong columns too.

After manual inspection, most csv files have this schema:
['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']

We'll call this list: **correct_headers**

### CORRECT HEADERS: 

<font size="4" color="green">['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']</font>

In [38]:
#Checking which csv files have differing header order
print("PROBLEM FILES")
#The majority of the .csv files have this schema:
correct_headers = ['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']
problem_files = []
print(f"\033[32mCORRECT HEADERS: \n{correct_headers}\033[0m") #green font to stand out
print("-" * 160)
print("-" * 160)
for f in csv_files:
    df = pd.read_csv(f, nrows=0)  
    if (list(df.columns)) != correct_headers:
        print(f"File: {f.name}")
        print(list(df.columns))
        print("-" * 160)
        problem_files.append(f.name)

PROBLEM FILES
CORRECT HEADERS: 
['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']
----------------------------------------------------------------------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------------------------------------------------------------------
File: 423JourneyDataExtract01Jul2025-15Jul2025.csv
['Number', 'Start date', 'Start station', 'Start station number', 'End date', 'End station', 'End station number', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']
----------------------------------------------------------------------------------------------------------------------------------------------------------------
File: 424JourneyDataExtract16Jul2025-31Jul2025.csv
['Number', 'S

In [39]:
print(f"Number of inconsistent files: {len(problem_files)}")

Number of inconsistent files: 11


We have 11 csv files with **inconsistent schema**. For each of these 11 csv files, we will need to realign their columns to match the **correct_headers**

Need to check if these csv files have the same feature names as the Correct headers list above. All header names must match for all 48 files.


In [48]:
#Need to check if these csv files have the same feature names as the Correct headers list above. All header names must match for all 48 files.
expected = set(correct_headers)
for f in csv_files:
    df = pd.read_csv(f, nrows=0)
    if (list(df.columns)) != correct_headers:
        print(f"File {f.name} has same Column headings? {set(df.columns) == expected}")
        print

File 423JourneyDataExtract01Jul2025-15Jul2025.csv has same Column headings? True
File 424JourneyDataExtract16Jul2025-31Jul2025.csv has same Column headings? True
File 425JourneyDataExtract01Aug2025-15Aug2025.csv has same Column headings? True
File 427JourneyDataExtract01Sep2025-15Sep2025.csv has same Column headings? True
File 428JourneyDataExtract16Sep2025-30Sep2025.csv has same Column headings? True
File 429JourneyDataExtract01Oct2025-15Oct2025.csv has same Column headings? True
File 430JourneyDataExtract16Oct2025-31Oct2025.csv has same Column headings? True
File 431JourneyDataExtract01Nov2025-15Nov2025.csv has same Column headings? True
File 432JourneyDataExtract16Nov2025-30Nov2025.csv has same Column headings? True
File 433JourneyDataExtract01Dec2025-15Dec2025.csv has same Column headings? True
File 434JourneyDataExtract16Dec2025-31Dec2025.csv has same Column headings? True


Because the headers have the same names and are just in a diffrent order, I can now re-order headers and write them as corrected csv's (NOT overwriting original files)

In [52]:
#example dataframe
df_1 = pd.DataFrame({
    "A": [1,2,3],
    "B": [4,5,6],
    "C": [7, 8, 9],
    "D": [10, 11, 12],
    "E": [13, 14, 15],
})

df_2 = pd.DataFrame({
    "B": [17,42,3],
    "A": [4,5,6],
    "D": [75, 8, 9],
    "C": [10, 131, 152],
    "E": [153, 14, 415],
})

In [53]:
my_list = ["A", "B", "C", "D", "E"]
expected = set(my_list)
has_same_cols_df1 = set(df_1.columns) == expected
has_same_cols_df2 = set(df_2.columns) == expected

print("df_1 matches:", has_same_cols_df1)  # True
print("df_2 matches:", has_same_cols_df2)  # True


df_1 matches: True
df_2 matches: True


In [56]:
new_order = ["E", "A", "C", "B", "D"]

In [57]:
df_reordered = df_1[new_order]

In [58]:
df_reordered


,E,A,C,B,D
0,13,1,7,4,10
1,14,2,8,5,11
2,15,3,9,6,12
